# Смертность по брендам ТС


In [1]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.io as pio

import data_loading
import visualize
import brands
import hypotheses as hyp

plt.style.use("default")
pio.renderers.default = "plotly_mimetype"
pd.set_option("display.max_columns", None)

FIGURES = Path("figures")
FIGURES.mkdir(exist_ok=True)

path = Path("../data/raw/vladimirskaia-oblast.geojson")
records = data_loading.load_records(path)
df = data_loading.record_to_full_frame(records)

occupants = brands.occupants_frame(df)
len(occupants)


53579

## Охват данных


In [2]:
display(pd.Series({
    "участников в ТС": len(occupants),
    "исход известен": int(occupants["status_known"].sum()),
    "исход не определён": int((~occupants["status_known"]).sum()),
    "бренд не указан": int(occupants["brand"].isna().sum()),
    "погибло": int(occupants["died"].sum()),
}))


участников в ТС       53579
исход известен        51655
исход не определён     1924
бренд не указан        3176
погибло                1982
dtype: int64

## Смертность по брендам


In [3]:
mortality = brands.brand_mortality(occupants)
display(mortality.style.format({
    "mortality": "{:.2%}", "median_year": "{:.0f}",
}))


,brand,occupants,died,median_year,mortality
0,CHEVROLET,1820,50,2009,2.75%
1,ГАЗ,2038,58,2011,2.85%
2,TOYOTA,1507,44,2008,2.92%
3,KIA,2566,79,2013,3.08%
4,NISSAN,1718,55,2010,3.20%
5,RENAULT,3198,104,2012,3.25%
6,VOLKSWAGEN,1862,65,2011,3.49%
7,HYUNDAI,3308,116,2011,3.51%
8,MERCEDES,1646,62,2001,3.77%
9,FORD,1706,66,2008,3.87%


## Гипотеза: освещение и смертность в ночных ДТП


In [4]:
night = hyp.night_accidents(df)

display(night.groupby("lit").agg(
    accidents=("fatal", "size"), fatal=("fatal", "sum"),
).rename(index={True: "освещение включено", False: "освещение отсутствует"})
 .assign(share=lambda t: t["fatal"] / t["accidents"])
 .style.format({"share": "{:.1%}"}))


,accidents,fatal,share
lit,,,
освещение отсутствует,2923,645,22.1%
освещение включено,4819,478,9.9%


### Сравнение внутри ячеек


In [5]:
cells = hyp.cell_lighting_table(night, min_each=5)

display(pd.Series({
    "ячеек в сравнении": len(cells),
    "ДТП в них": int(cells[["accidents_lit", "accidents_unlit"]].to_numpy().sum()),
    "из ночных ДТП всего": len(night),
    "ячеек, где без освещения смертельнее": int((cells["difference"] > 0).sum()),
}))
display(cells.style.format({
    "share_lit": "{:.1%}", "share_unlit": "{:.1%}", "difference": "{:+.1%}",
}))


ячеек в сравнении                         84
ДТП в них                               5761
из ночных ДТП всего                     7742
ячеек, где без освещения смертельнее      57
dtype: int64

,accidents_unlit,accidents_lit,fatal_unlit,fatal_lit,share_lit,share_unlit,difference
cell,,,,,,,
861188aafffffff,12,6,3,4,66.7%,25.0%,-41.7%
8611883afffffff,14,7,1,3,42.9%,7.1%,-35.7%
86118951fffffff,11,7,1,3,42.9%,9.1%,-33.8%
86118b8b7ffffff,20,11,2,4,36.4%,10.0%,-26.4%
8611ab41fffffff,12,13,0,3,23.1%,0.0%,-23.1%
861188907ffffff,21,5,4,2,40.0%,19.0%,-21.0%
86118cb1fffffff,13,21,1,6,28.6%,7.7%,-20.9%
86118954fffffff,8,6,1,2,33.3%,12.5%,-20.8%
86118890fffffff,17,61,0,12,19.7%,0.0%,-19.7%


## Гипотеза: смертность в ВАЗ при одинаковых обстоятельствах ДТП


In [ ]:
mixed = brands.mixed_brand_occupants(df)

display(pd.Series({
    "ДТП с ВАЗ и другой маркой": mixed["source_id"].nunique(),
    "человек в них": len(mixed),
}))
display(mixed.groupby("target")
    .agg(people=("died", "size"), died=("died", "sum"))
    .rename(index={True: "в ВАЗ", False: "в другой марке"})
    .assign(share=lambda t: t["died"] / t["people"])
    .style.format({"share": "{:.2%}"}))


### Разница внутри каждого ДТП


In [ ]:
pairs = brands.accident_brand_table(mixed)

display(pd.Series({
    "ДТП в сравнении": len(pairs),
    "в ВАЗ погибло чаще": int((pairs["difference"] > 0).sum()),
    "в другой марке чаще": int((pairs["difference"] < 0).sum()),
    "исход одинаковый": int((pairs["difference"] == 0).sum()),
}))
display(pairs.head())


### Перестановочный тест и доверительный интервал


In [ ]:
perm = brands.brand_permutation(mixed)
boot = brands.brand_bootstrap(pairs)

display(pd.Series({
    "эффект, п.п.": round(boot["observed"] * 100, 2),
    "95% ДИ снизу": round(boot["ci_lower"] * 100, 2),
    "95% ДИ сверху": round(boot["ci_upper"] * 100, 2),
    "p-value": perm["p_value"] or f"< {1 / perm['draws']}",
    "ДТП": perm["accidents"],
    "перестановок / повторов": perm["draws"],
    "seed": perm["seed"],
}))


## Гипотеза 3: стаж водителя и вероятность пострадать


In [ ]:
drivers = hyp.driver_outcomes(df)
chi2 = hyp.experience_chi2(drivers)
diff = hyp.experience_difference(drivers)

display(hyp.experience_table(drivers).style.format({"share": "{:.1%}"}))
display(pd.Series({
    "хи-квадрат": round(chi2["chi2"], 1),
    "df": chi2["dof"],
    "p-value": chi2["p_value"],
    "V Крамера": round(chi2["cramers_v"], 3),
    "минимальная ожидаемая частота": round(chi2["min_expected"]),
    "монотонное убывание": chi2["monotone"],
    "разница крайних групп, п.п.": round(diff["difference"] * 100, 1),
    "95% ДИ снизу": round(diff["ci_lower"] * 100, 1),
    "95% ДИ сверху": round(diff["ci_upper"] * 100, 1),
}))
